# Agent Development Kit（ADK 2.x）によるグラフワークフローの構築

## 概要

複雑なエージェントアプリケーションでは、線形なエージェントチェーンや単純なサブエージェントルーティングでは十分な柔軟性が得られないことがよくあります。Google Agent Development Kit（ADK）2.x では**グラフワークフロー**（`Workflow`）が導入され、条件付きルーティング、並列処理、反復型リファインメントループ、Human-in-the-Loop（人間による介入）などを含む非線形な実行フローが可能になりました。

このノートブックでは、ADK 2.x を使用して 4 つの一般的なグラフワークフローパターンを構築する方法を学びます。

1. **並列処理ワークフロー（`JoinNode`）**: 複数の処理ノードを並列に実行し、同期ノードを使用してそれらの結果を集約します。
2. **Evaluator-Optimizer リファインメントループ**: エバリュエーターエージェントが生成されたコンテンツを評価し、品質基準を満たすまでフィードバックを伴ってループバックする反復ループを構築します。
3. **ルーティングとブランチングワークフロー**: LLM エージェントを使用して受信リクエストを分類し、専用のハンドラーノードに動的にルーティングします（単一またはマルチブランチの並列実行をサポート）。
4. **Human-in-the-Loop（HITL）ワークフロー**: `RequestInput` を使用してワークフローの実行を一時中断し、再開または分岐する前に人間のレビューやフィードバックを求めます。

各ワークフローは `%%writefile` を使用してディスクに書き込まれ、**ADK Developer UI**（`adk web`）を使用してインタラクティブにテストされます。

## 環境のセットアップと設定

まず、Google Cloud Vertex AI と ADK で必要な環境変数を設定します。

In [ ]:
import os

LOCATION = "global"
os.environ["GOOGLE_CLOUD_LOCATION"] = LOCATION
os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "TRUE"

In [ ]:
%%bash
mkdir -p adk_workflows
echo > adk_workflows/.env "GOOGLE_CLOUD_LOCATION=$GOOGLE_CLOUD_LOCATION
GOOGLE_GENAI_USE_VERTEXAI=$GOOGLE_GENAI_USE_VERTEXAI"


---
## 1. 並列処理ワークフロー（`JoinNode`）

並列ワークフローを使用すると、複数の独立したタスクを同時に実行し、すべてのブランチの完了を待ってから、結合された状態（state）を下流の集約ノードに渡すことができます。

### ワークフローのアーキテクチャ
- **並列開始ノード（`node_A`、`node_B`、`node_C`）**: `START` からトリガーされた際に並行して実行される Python ノード。
- **同期ノード（`JoinNode`）**: すべての受信並列エッジの完了を待機します。
- **集約ノード（`node_D`）**: 収集された状態キー（`node_A`、`node_B`、`node_C`）を受け取り、最終的なサマリーをフォーマットします。

In [ ]:
!mkdir -p ./adk_workflows/graph1_parallel

In [ ]:
%%writefile ./adk_workflows/graph1_parallel/agent.py
"""ADK 2.x 並列処理ワークフロー。"""

# pylint: disable=invalid-name

from google.adk import Event, Workflow
from google.adk.workflow import JoinNode


# 1. 並列処理ノードの定義（A、B、C）
def node_A(node_input: str):
    return Event(
        message=f"Node A executed... input={node_input}",
        output=int(node_input),
    )


def node_B(node_input: str):
    input_b = int(node_input)
    return Event(
        message=f"Node B executed... input={node_input}",
        output=input_b * 100,
    )


def node_C(node_input: str):
    input_c = int(node_input)
    return Event(
        message=f"Node C executed... input={node_input}",
        output=input_c * input_c,
    )


# 2. 集約および表示ノードの定義（D）
def node_D(node_input: dict) -> Event:
    """JoinNode から出力を収集し、合計を計算して表示します。"""
    val_a = node_input.get("node_A", 0.0)
    val_b = node_input.get("node_B", 0.0)
    val_c = node_input.get("node_C", 0.0)

    total_sum = val_a + val_b + val_c

    display_message = (
        f"### Execution Complete!\n\n"
        f"Successfully collected parallel outputs:\n"
        f"- **Node A Output:** `{val_a}`\n"
        f"- **Node B Output:** `{val_b}`\n"
        f"- **Node C Output:** `{val_c}`\n\n"
        f"---\n"
        f"### Result\n"
        f"**Node D (Total Sum):** `{total_sum}`"
    )

    return Event(message=display_message, output=total_sum)


join_node = JoinNode(name="join_node")

# 3. ワークフローの構築
root_agent = Workflow(
    name="parallel_workflow",
    edges=[
        ("START", node_A, join_node),
        ("START", node_B, join_node),
        ("START", node_C, join_node),
        (join_node, node_D),
    ],
)


### ADK Developer UI を開く

`adk web` コマンドを使用して ADK Dev UI を起動し、エージェントとインタラクティブに対話できます。

以下のコマンドを実行し、生成された Web リンクをクリックしてインターフェースを開きます。

**注:** `adk web` は継続して実行されます。UI でのこのワークフローのテストが完了したら、次のセクションに進む前にセルを停止 / 割り込み（■）してください。

In [ ]:
# Cloud Workstations の場合
!adk web adk_workflows --allow_origins "regex:https://.*\.cloudworkstations\.dev"


**注:** Agent Platform Workbench を使用している場合は、コメントアウトを解除して以下のセルを実行してください。

In [ ]:
# %%bash
# PROXY_BASE=$(curl -s http://metadata.google.internal/computeMetadata/v1/instance/attributes/proxy-url -H "Metadata-Flavor: Google")
# echo "--------------------------------------------------------"
# echo "🔗 ACCESS HERE: https://${PROXY_BASE}/proxy/8000/dev-ui/"
# echo "--------------------------------------------------------"
# adk web adk_workflows --url_prefix /proxy/8000  --allow_origins "regex:https://.*\.notebooks\.googleusercontent\.com"

### Developer UI の使用
左上のドロップダウンで `graph1_parallel` を選択し、以下の手順に沿ってクエリを送信します。
- 初期値を入力します（例: `"5"`）。
- ノード A、B、C が並列に実行され、`JoinNode` がそれらの状態変数を収集してノード D が集約する様子を確認します。
- 完了したら、セクション 2 に進む前に実行中の `adk web` セルを停止 / 割り込み（■）してください。

---
## 2. Evaluator-Optimizer リファインメントループ

反復型リファインメントループは、ジェネレーター（生成担当）の出力を定義された制約や品質メトリクスに照らして評価し、基準が満たされるまでフィードバックをジェネレーターに戻します。

### ワークフローのアーキテクチャ
- **入力プロセッサ（`process_input`）**: ユーザーの初期トピックを共有ワークフロー状態（`{topic}`）に保存します。
- **ジェネレーターエージェント（`generate_headline`）**: 状態変数 `{topic}` およびオプションの `{feedback?}` を取り入れて見出しを作成します。
- **エバリュエーターエージェント（`evaluate_headline`）**: 構造化された Pydantic スキーマ（`Feedback`）を使用して、見出しが `"tech-related"`（テクノロジー関連）か `"unrelated"`（無関係）かを判定します。
- **ルーターノード（`route_headline`）**: `Event(route=feedback.grade)` を発行します。`"unrelated"` の場合、エッジは `generate_headline` にループバックします。

In [ ]:
!mkdir -p ./adk_workflows/graph2_loop

In [ ]:
%%writefile ./adk_workflows/graph2_loop/agent.py
"""ADK 2.x Evaluator-Optimizer リファインメントループワークフロー。"""

from typing import Literal

from google.adk import Agent, Event, Workflow
from pydantic import BaseModel, Field

MODEL = "gemini-3.5-flash"


class Feedback(BaseModel):
    grade: Literal["tech-related", "unrelated"] = Field(
        description=(
            "Decide if the headline is related to technology or software "
            "engineering."
        ),
    )
    feedback: str = Field(
        description=(
            "If the headline is unrelated to technology, provide feedback "
            "on how to make it more tech-focused."
        ),
    )


def process_input(node_input: str):
    """ユーザー入力を state に格納します。"""
    return Event(state={"topic": node_input})


generate_headline = Agent(
    name="generate_headline",
    model=MODEL,
    instruction="""
    Write a headline about the topic "{topic}".
    If feedback is provided, take it into account.
    The feedback: {feedback?}
    """,
)

evaluate_headline = Agent(
    name="evaluate_headline",
    model=MODEL,
    instruction="""
    Grade whether the headline is related to technology or software engineering.
    """,
    output_schema=Feedback,
    output_key="feedback",
)


def route_headline(node_input: Feedback):
    return Event(route=node_input.grade)


root_agent = Workflow(
    name="evaluator_optimizer_loop",
    edges=[
        (
            "START",
            process_input,
            generate_headline,
            evaluate_headline,
            route_headline,
        ),
        # リファインメントループ:
        (route_headline, {"unrelated": generate_headline}),
    ],
)


### ADK Developer UI を開く

`adk web` コマンドを使用して ADK Dev UI を起動し、エージェントとインタラクティブに対話できます。

以下のコマンドを実行し、生成された Web リンクをクリックしてインターフェースを開きます。

**注:** `adk web` は継続して実行されます。UI でのこのワークフローのテストが完了したら、次のセクションに進む前にセルを停止 / 割り込み（■）してください。

In [ ]:
# Cloud Workstations の場合
!adk web adk_workflows --allow_origins "regex:https://.*\.cloudworkstations\.dev"


**注:** Agent Platform Workbench を使用している場合は、コメントアウトを解除して以下のセルを実行してください。

In [ ]:
# %%bash
# PROXY_BASE=$(curl -s http://metadata.google.internal/computeMetadata/v1/instance/attributes/proxy-url -H "Metadata-Flavor: Google")
# echo "--------------------------------------------------------"
# echo "🔗 ACCESS HERE: https://${PROXY_BASE}/proxy/8000/dev-ui/"
# echo "--------------------------------------------------------"
# adk web adk_workflows --url_prefix /proxy/8000  --allow_origins "regex:https://.*\.notebooks\.googleusercontent\.com"

### Developer UI の使用
左上のドロップダウンで `graph2_loop` を選択し、以下の手順に沿ってクエリを送信します。
- 非技術的なトピックを入力します（例: `"The history of pizza"`（ピザの歴史））。
- エバリュエーターがドラフトを `unrelated` と判定し、フィードバックを伴うリファインメントループをトリガーして、基準を満たすまでジェネレーターが再ドラフトを行う様子を確認します。
- 技術的なトピックを入力し（例: `"Recent advancements in quantum computing"`（量子コンピューティングの最近の進歩））、初回で合格することを確認します。
- 完了したら、セクション 3 に進む前に実行中の `adk web` セルを停止 / 割り込み（■）してください。

---
## 3. ルーティングとブランチングワークフロー

ルーティングワークフローでは、受信したリクエストが 1 つ以上のカテゴリに分類され、実行が適切な宛先ノードに動的にルーティングされます。複数のカテゴリが特定された場合、ADK は対象のブランチを自動的に並列実行します。

### ワークフローのアーキテクチャ
- **分類ノード（`process_message`）**: ユーザー入力を分析し、カテゴリ（`BUG`、`CUSTOMER_SUPPORT`、`LOGISTICS`）を出力する LLM エージェント（`gemini-3.5-flash`）。
- **ルーターノード（`router`）**: LLM の出力をルートに分割し、`Event(route=...)` を発行する Python 関数。
- **宛先ノード**: 特定の Python ハンドラー関数（`response_1_bug`、`response_2_support`、`response_3_logistics`）。

In [ ]:
!mkdir -p ./adk_workflows/graph3_router

In [ ]:
%%writefile ./adk_workflows/graph3_router/agent.py
"""ADK 2.x ルーティングワークフロー。"""

from google.adk import Agent, Event, Workflow

MODEL = "gemini-3.5-flash"

# 1. LLM 分類エージェント
process_message = Agent(
    name="process_message",
    model=MODEL,
    instruction="""Classify user message into either "BUG", "CUSTOMER_SUPPORT",
     or "LOGISTICS". If you think a message applies to more than one category,
     reply with a comma separated list of categories.
  """,
    output_schema=str,
)


# 2. ルーターノード
def router(node_input: str):
    # LLM からのカンマ区切り文字列を分割
    routes = node_input.split(",")
    # 空白文字をクリーンアップ
    routes = [route.strip() for route in routes]

    # 'route' を指定して Event を発行することで、次に走査するエッジを指定
    return Event(route=routes)


# 3. 宛先ノード
def response_1_bug():
    return Event(message="Handling bug...")


def response_2_support():
    return Event(message="Handling customer support...")


def response_3_logistics():
    return Event(message="Handling logistics...")


# 4. ワークフローの構築
root_agent = Workflow(
    name="routing_workflow",
    edges=[
        # 1. LLM 分類器から実行を開始し、出力をルーターに渡す
        ("START", process_message, router),
        # 2. ルーターから発行されたルートを宛先ノードにマッピング
        (
            router,
            {
                "BUG": response_1_bug,
                "CUSTOMER_SUPPORT": response_2_support,
                "LOGISTICS": response_3_logistics,
            },
        ),
    ],
)


### ADK Developer UI を開く

`adk web` コマンドを使用して ADK Dev UI を起動し、エージェントとインタラクティブに対話できます。

以下のコマンドを実行し、生成された Web リンクをクリックしてインターフェースを開きます。

**注:** `adk web` は継続して実行されます。UI でのこのワークフローのテストが完了したら、次のセクションに進む前にセルを停止 / 割り込み（■）してください。

In [ ]:
# Cloud Workstations の場合
!adk web adk_workflows --allow_origins "regex:https://.*\.cloudworkstations\.dev"


**注:** Agent Platform Workbench を使用している場合は、コメントアウトを解除して以下のセルを実行してください。

In [ ]:
# %%bash
# PROXY_BASE=$(curl -s http://metadata.google.internal/computeMetadata/v1/instance/attributes/proxy-url -H "Metadata-Flavor: Google")
# echo "--------------------------------------------------------"
# echo "🔗 ACCESS HERE: https://${PROXY_BASE}/proxy/8000/dev-ui/"
# echo "--------------------------------------------------------"
# adk web adk_workflows --url_prefix /proxy/8000  --allow_origins "regex:https://.*\.notebooks\.googleusercontent\.com"

### Developer UI の使用
左上のドロップダウンで `graph3_router` を選択し、以下の手順に沿ってクエリを送信します。
- 複数のカテゴリにまたがるクエリを送信してみます（例: `"I can't track my shipment, and the mobile app keeps crashing!"`（荷物の追跡ができず、モバイルアプリがクラッシュし続けます！））。
- **BUG** と **LOGISTICS** の両方の宛先ノードが実行されることを確認します。
- 完了したら、セクション 4 に進む前に実行中の `adk web` セルを停止 / 割り込み（■）してください。

---
## 4. Human-in-the-Loop（HITL）ワークフロー

ワークフローでは、取り返しのつかないアクションを実行する前に、人間の承認や介入が必要になることがよくあります。ADK では、`RequestInput` オブジェクトを yield することで、ユーザーが入力を提供するかフィードバックを与えるまでワークフローの実行を一時停止します。

### ワークフローのアーキテクチャ
- **入力プロセッサ（`process_input`）**: 顧客からのクレームと空のフィードバックでワークフローの状態を初期化します。
- **ドラフトメールエージェント（`draft_email`）**: `{complaint}` と `{feedback?}` に基づいてカスタマーサービスの返信の下書きを作成する LLM エージェント。
- **レビュー要求ノード（`request_human_review`）**: `RequestInput` を yield し、実行を一時停止して人間のレビュー担当者にプロンプトを表示します。
- **レビュー処理ルーター（`handle_human_review`）**: 人間の応答を評価します: `"approve"` -> `send_email`、`"reject"` -> `reject_email`、または修正フィードバック -> `draft_email` にループバック。

In [ ]:
!mkdir -p ./adk_workflows/graph4_hitl

In [ ]:
%%writefile ./adk_workflows/graph4_hitl/agent.py
"""ADK 2.x Human-in-the-Loop ワークフロー。"""

from google.adk import Agent, Event, Workflow
from google.adk.events import RequestInput

MODEL = "gemini-3.5-flash"


def process_input(node_input: str):
    """初期の顧客のクレームを入力として受け取り、state を設定します。"""
    return Event(state={"complaint": node_input, "feedback": ""})


draft_email = Agent(
    name="draft_email",
    model=MODEL,
    instruction="""
    Please write a polite, helpful response email to customer complaint:
    "{complaint}"
    If there is feedback from the manager to revise draft, incorporate it:
    "{feedback?}"
    """,
    output_key="draft",
)


def request_human_review(draft: str):
    return RequestInput(
        message=(
            "Please review the following draft email and provide "
            f"'approve', 'reject', or feedback to revise.\n\n---\n{draft}\n---"
        )
    )


def handle_human_review(node_input: str):
    user_resp = node_input.lower().strip()
    if user_resp == "reject":
        return Event(route="rejected")
    if user_resp == "approve":
        return Event(route="approved")
    return Event(state={"feedback": node_input}, route="revise")


def reject_email():
    return Event(message="Draft rejected.")


def send_email(draft: str):
    # pylint: disable=unused-argument
    return Event(message="Draft approved and sent successfully.")


root_agent = Workflow(
    name="hitl_workflow",
    edges=[
        (
            "START",
            process_input,
            draft_email,
            request_human_review,
            handle_human_review,
        ),
        (
            handle_human_review,
            {
                "revise": draft_email,
                "approved": send_email,
                "rejected": reject_email,
            },
        ),
    ],
)


### ADK Developer UI を開く

`adk web` コマンドを使用して ADK Dev UI を起動し、エージェントとインタラクティブに対話できます。

以下のコマンドを実行し、生成された Web リンクをクリックしてインターフェースを開きます。

**注:** `adk web` は継続して実行されます。UI でのこのワークフローのテストが完了したら、セルを停止 / 割り込み（■）してください。

In [ ]:
# Cloud Workstations の場合
!adk web adk_workflows --allow_origins "regex:https://.*\.cloudworkstations\.dev"


**注:** Agent Platform Workbench を使用している場合は、コメントアウトを解除して以下のセルを実行してください。

In [ ]:
# %%bash
# PROXY_BASE=$(curl -s http://metadata.google.internal/computeMetadata/v1/instance/attributes/proxy-url -H "Metadata-Flavor: Google")
# echo "--------------------------------------------------------"
# echo "🔗 ACCESS HERE: https://${PROXY_BASE}/proxy/8000/dev-ui/"
# echo "--------------------------------------------------------"
# adk web adk_workflows --url_prefix /proxy/8000  --allow_origins "regex:https://.*\.notebooks\.googleusercontent\.com"

### Developer UI の使用
左上のドロップダウンで `graph4_hitl` を選択し、以下の手順に沿ってクエリを送信します。
1. 初期のクレームを送信します（例: `"My delivery was 3 days late and the box was crushed!"`（配達が3日遅れたうえに、箱がつぶれていました！））。
2. ワークフローの実行が一時停止し、人間のレビュー用のプロンプトが表示されることを確認します。
3. UI に修正のフィードバックを入力して（例: `"Too formal, make it shorter and offer a 10% discount."`（堅苦しすぎるので、もっと短くして10%の割引を提示してください））、ワークフローが下書きを更新する様子を確認します。
4. `"approve"` または `"reject"` と返信してワークフローを完了します。
- 完了したら、実行中の `adk web` セルを停止 / 割り込み（■）してください。

---
## 5. チャレンジラボ: 反復型エッセイ執筆ワークフロー

このチャレンジラボでは、ADK 2.x の**グラフワークフロー**（`Workflow`）を使用して、エンドツーエンドのマルチエージェントによるエッセイの執筆と推敲（リファインメント）を行うワークフローを設計・実装します。

### ワークフローのアーキテクチャと設計

このワークフローは、専門エージェントを連携させて、任意のトピックに関するエッセイを反復的に調査、執筆、批評、推敲します。

1. **入力処理（`process_input`）**: ユーザーの初期入力トピックを取得し、セッション状態（state）のキー `"topic"` に保存します。
2. **計画（`planner_agent`）**: **Google 検索**（`google_search`）を使用して関連するテーマや論点を調査し、構造化されたエッセイのアウトラインを生成して、状態キー `"plan"` に保存します。
3. **執筆と調査（`writer_agent`）**: アウトラインと批評フィードバックに基づいてエッセイを下書きします。事実の裏付けや根拠を得るために **Google 検索**（`google_search`）を使用し、下書きを状態キー `"draft"` に保存します。
4. **批評と評価（`evaluator_agent`）**: 学術レビュアーエージェントが、品質と構成の基準に照らしてエッセイを採点し、構造化された評価（`CritiqueFeedback`）を返します。
5. **反復的推敲（`route_critique`）**: 評価グレードを検証します。修正が必要であり、修正制限内（`MAX_REVISIONS = 2`）である場合は、批評フィードバックとともに `writer_agent` にルーティングします。それ以外の場合は、`finalize_essay` にルーティングします。
6. **最終仕上げ（`finalize_essay`）**: 完成したエッセイをフォーマットして出力します。

#### アーキテクチャ図

```mermaid
flowchart TD
    START([START]) --> ProcessInput["1. process_input<br/>(Store input in state: 'topic')"]
    ProcessInput --> Planner["2. planner_agent<br/>(Generate outline with Google Search &rarr; plan)"]
    Planner --> Writer["3. writer_agent<br/>(Draft/revise essay with Google Search &rarr; draft)"]
    Writer --> Evaluator["4. evaluator_agent<br/>(Critique draft & assign grade &rarr; critique_result)"]
    Evaluator --> RouteCritique{"5. route_critique<br/>(Check grade & max revisions)"}
    RouteCritique -- "route: 'revise'<br/>(revision_count < 2)" --> Writer
    RouteCritique -- "route: 'accept'<br/>(accepted or max revisions reached)" --> Finalize["6. finalize_essay<br/>(Display final essay)"]
    Finalize --> END([END])
```

#### 共有ワークフロー状態（State）

`essay_writing_workflow` 内では、ノードは共有セッション状態を介して通信します。

| 状態キー | 型 | 説明 | 更新元 |
|---|---|---|---|
| `topic` | `str` | エッセイのユーザー入力トピック。 | `process_input` |
| `plan` | `str` | セクションの見出しと論点を含む上位レベルのアウトライン。 | `planner_agent` (`output_key="plan"`) |
| `draft` | `str` | エッセイの現在の Markdown 下書き。 | `writer_agent` (`output_key="draft"`) |
| `critique_result` | `CritiqueFeedback` | `grade`（'accept'/'revise'）と詳細な `critique` を含む構造化評価。 | `evaluator_agent` (`output_key="critique_result"`) |
| `critique` | `str` | 修正中に執筆者のプロンプトに渡されるテキストフィードバック。 | `route_critique` |
| `revision_count` | `int` | 推敲ループの反復回数を追跡するカウンター。 | `route_critique`（ループごとにインクリメント） |

#### グラフエッジの仕様

ワークフローは以下の構成でノードを接続します。
1. **線形パイプラインシーケンス**:
   `("START", process_input, planner_agent, writer_agent, evaluator_agent, route_critique)`
2. **条件分岐とループ**:
   `(route_critique, {"revise": writer_agent, "accept": finalize_essay})`

### 演習: `adk_workflows/graph5_challenge_essay/agent.py` の完成

エディタでスターターテンプレートファイルを直接開きます。
- [`adk_workflows/graph5_challenge_essay/agent.py`](file://./adk_workflows/graph5_challenge_essay/agent.py)

上記のアーキテクチャ仕様に従って、`agent.py` の実装を完成させてください。
1. **入力処理ノード（`process_input`）**:
   - `Event(state={"topic": node_input})` を返して、受信したトピック文字列をセッション状態に保存し、下流のエージェントが `{topic}` を参照できるようにします。
2. **専門エージェント**:
   - `planner_agent`: `gemini-3.8-flash` と `tools=[google_search]` で構成され、トピックを調査して構造化されたエッセイのアウトラインを作成し、`output_key="plan"` に保存します。
   - `writer_agent`: `gemini-3.8-flash` と `tools=[google_search]` で構成されます。`{plan}` およびオプションの `{critique?}` を取り入れてエッセイの生成 / 推敲を行い、`output_key="draft"` に保存します。
   - `evaluator_agent`: `gemini-3.8-flash` で構成され、厳格な教授として `output_schema=CritiqueFeedback` および `output_key="critique_result"` を使用して `{draft}` を採点します。初稿がグレード `'revise'` を受けて推敲ループがトリガーされるよう、学術的な基準を高く設定します（具体的な実証的証拠、詳細な実際のケーススタディ、および深い反論を要求します）。
3. **ルーターノード（`route_critique`）**:
   - `node_input.grade` と `revision_count` を検査します:
   - `grade == "revise"` かつ `revision_count < MAX_REVISIONS`（2）の場合: `revision_count` をインクリメントし、批評を state に保存して、`Event(route="revise")` を発行します。
   - それ以外の場合: `Event(route="accept")` を発行します。
4. **ワークフローの構築（`root_agent`）**:
   - 線形シーケンスと推敲ループを接続するエッジを持つ `"essay_writing_workflow"` という名前の `Workflow` を構築します。

> **注**: 比較のために、参考解答が [`adk_workflows/graph5_challenge_essay/solution.py`](file://./adk_workflows/graph5_challenge_essay/solution.py) に用意されています。

### ADK Developer UI を開く

`adk_workflows/graph5_challenge_essay/agent.py` を更新して保存した後、`adk web` コマンドを使用して ADK Dev UI を起動し、ワークフローをテストします。

以下のコマンドを実行し、生成された Web リンクをクリックしてインターフェースを開きます。

**注:** `adk web` は継続して実行されます。UI でのこのワークフローのテストが完了したら、セルを停止 / 割り込み（■）してください。

In [ ]:
# Cloud Workstations の場合
!adk web adk_workflows --allow_origins "regex:https://.*\.cloudworkstations\.dev"


**注:** Agent Platform Workbench を使用している場合は、コメントアウトを解除して以下のセルを実行してください。

In [ ]:
# %%bash
# PROXY_BASE=$(curl -s http://metadata.google.internal/computeMetadata/v1/instance/attributes/proxy-url -H "Metadata-Flavor: Google")
# echo "--------------------------------------------------------"
# echo "🔗 ACCESS HERE: https://${PROXY_BASE}/proxy/8000/dev-ui/"
# echo "--------------------------------------------------------"
# adk web adk_workflows --url_prefix /proxy/8000  --allow_origins "regex:https://.*\.notebooks\.googleusercontent\.com"

### Developer UI の使用

左上のエージェントのドロップダウンから `graph5_challenge_essay` を選択し、ワークフローをテストします。
1. ADK Web がワークフローのインタラクティブな視覚的 DAG ダイアグラムをどのようにレンダリングするかを確認します。
2. 入力ボックスにエッセイのトピックを入力します（例: `"The ethical implications of artificial intelligence in autonomous vehicles"`（自動運転車における人工知能の倫理的影響））。
3. グラフノードを通過するリアルタイムの実行フローを確認します。
   - `process_input` がトピックを state に保存します。
   - `planner_agent` が Google 検索で調査し、アウトラインを生成します。
   - `writer_agent` がエッセイを下書きします。
   - `evaluator_agent` が下書きを批評します。
   - `route_critique` は、修正が必要な場合はループバックし、承認されたら `finalize_essay` に分岐します。
4. 完了したら、実行中の `adk web` セルを停止 / 割り込み（■）してください。

Copyright 2026 Google LLC

Licensed under the Apache License, Version 2.0 (the "License"); you may not use this file except in compliance with the License. You may obtain a copy of the License at

    https://www.apache.org/licenses/LICENSE-2.0

Unless required by applicable law or agreed to in writing, software distributed under the License is distributed on an "AS IS" BASIS, WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied. See the License for the specific language governing permissions and limitations under the License.